<a href="https://colab.research.google.com/github/DeKUT-DSAIL/MasharikiWeather/blob/main/docs/source/examples/masharikiweather_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MasharikiWeather Demo
This is an internal (private) Weather team (at DSAIL) notebook to demonstrate the progress, potential and intent of MasharikiWeather dataset.

This is the progress of the first version, it will be iteratively improved with time.

## Setting up
One requires the huggingface dataset token to be able to access the data.

These are measures that have been put up because we do not have the rights to repackage some of the data products.


In [1]:
import os
from google.colab import userdata

In [2]:
mashariki_hf_token = userdata.get('MASHARIKI_HF_TOKEN') # Huggingface dataset token

In [5]:
!pip install masharikiweather --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.7 MB/s eta 0:00:00


In [6]:
# Log in to HF
from huggingface_hub import notebook_login
notebook_login()

In [7]:
import xarray as xr
import pandas as pd
import numpy as np
import zarr
import fsspec

from filter_stations import RetrieveData

In [ ]:
# GhCNd stations
ghcnd_station_list_ea = [
    'KE000063612', 'KE000063619', 'KE000063624',
    'KE000063661', 'KE000063723', 'KE000063740',
    'KE000063820', 'KEM00063686','KEM00063741',
    'KEM00063799', 'MZM00067205', 'RWM00064387',
    'SU000062941', 'TZ000063756', 'TZ000063832',
    'TZ000063862','TZ000063894', 'TZ000063962',
    'TZM00063870','ZA000067475']
    # 'TZ000063971' - Ignore Mtwara Airport station


In [8]:
from masharikiweather.dataset import MasharikiWeatherDataset

# Data Understanding
- Observations with their metadata (latitude, longitude, static priors)
- `Nearest Neighbor` and `Bilinear` interpolation methods of gridded products and station data.
- Demonstration of window generation for downstream ML tasks.

**N/B** Only precipitation is currently available for the station data but other variables will soon be included.

In [9]:
# Init the Dataset class
ds = MasharikiWeatherDataset(
    repo_id="DeKUT-DSAIL/weather-data",
    token=mashariki_hf_token,
    parameters=['total_precipitation'],
    source_obs=["tahmo", "ghcnd"],
    freq="h",
    years=[2020, 2021, 2022, 2023, 2024]
)

print("--- Station Metadata ---")

# Get the station metadata
metadata_df = ds.get_stations_metadata()
display(metadata_df)

(…)/topography/east_africa_static_priors.nc:   0%|          | 0.00/117M [00:00<?, ?B/s]

--- Station Metadata ---


,latitude,longitude,network,LandCover,Aspect,dem,Slope,DistanceToWater,STD_2000M,TPI_2000M,WE_DERIVATIVE_2000M_SIGRATIO1,SN_DERIVATIVE_2000M_SIGRATIO1,ASPECT_2000M_SIGRATIO1,SLOPE_2000M_SIGRATIO1,STD_10000M,TPI_10000M,WE_DERIVATIVE_10000M_SIGRATIO1,SN_DERIVATIVE_10000M_SIGRATIO1,ASPECT_10000M_SIGRATIO1,SLOPE_10000M_SIGRATIO1
station_id,,,,,,,,,,,,,,,,,,,,
TA00001,-1.123283,34.397991,tahmo,40.0,175.425201,1391.364319,0.674847,4333.137484,12.446386,2.286049,-0.002605,-0.004192,192.247016,0.675333,45.097866,-5.144433,-0.003341,0.008445,184.093083,1.136435
TA00019,-0.524114,34.255646,tahmo,40.0,121.096237,1292.409500,4.804829,6663.363765,82.371790,-7.081631,0.055322,-0.027170,105.017959,4.029285,128.660578,28.034848,0.004539,0.011223,168.387815,3.128075
TA00020,-1.653356,36.862396,tahmo,30.0,90.000000,1642.085512,0.381284,18839.528693,7.280683,-1.212345,0.005305,-0.001331,101.708727,0.380757,26.333211,-1.495561,0.005029,0.000915,105.636945,0.484817
TA00021,0.757986,35.173710,tahmo,40.0,266.820435,1954.092370,0.518028,5741.440373,17.723127,7.163968,-0.015837,0.006406,271.028880,1.181581,60.459173,18.949949,-0.009641,0.002271,260.886385,0.894991
TA00023,-2.388550,38.040768,tahmo,40.0,79.517418,785.137357,0.771984,79689.569136,14.194454,-2.124790,0.010299,0.006165,77.248711,0.782003,58.707464,-30.821983,0.010826,0.003948,89.098411,0.858292
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TZ000063894,-6.867000,39.200000,ghcnd,50.0,45.206219,58.099468,0.254812,8779.891930,4.624157,0.957107,0.004982,0.000164,83.451710,0.348581,34.528696,-8.148431,0.006123,-0.001121,125.651843,0.557380
TZ000063962,-10.667000,35.583000,ghcnd,40.0,303.696808,1031.006137,0.828336,43770.292325,14.287122,-9.760028,-0.010345,-0.000196,268.304293,0.857702,55.284006,-36.569998,-0.008069,0.002494,215.127311,1.344565
TZ000063971,-10.267000,40.183000,ghcnd,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [10]:
# Retrieve Ground Truth
obs_df, obs_mask = ds.get_observations(
    parameters=['total_precipitation'],
    return_mask=True
)

display(obs_df.head())

station_id,KE000063612,KE000063619,KE000063624,KE000063661,KE000063723,KE000063740,KE000063820,KEM00063686,KEM00063741,KEM00063799,...,TD00004,TD00005,TZ000063756,TZ000063832,TZ000063862,TZ000063894,TZ000063962,TZ000063971,TZM00063870,ZA000067475
parameter,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,...,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation
time,,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2020-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 02:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 03:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 04:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


For the GhCNd stations, the valid value is at timestamp `00:00` of each available day. The other values are filled values (with zero), one can change the filling method. There is a binary mask to show the missingness, this mask would help in any model training, it would effectively tell the model to ignore the padded zeros, calculating its loss over the valid observations only.

In [12]:
MasharikiWeatherDataset.get_gridded_for_stations?

In [13]:
# Retrieve Satellite/Reanalysis Context
print("\n--- Extracting Gridded Context ---")
gridded_context = ds.get_gridded_for_stations(
    groups=["era5", "imerg", "chirps", "tamsat"],
    years=[2024],
    variables=['total_precipitation']
)


--- Extracting Gridded Context ---


Extracting tamsat: 100%|██████████| 1/1 [00:11<00:00, 11.67s/it]


In [14]:
# preview era5
gridded_context['era5'].head()

station_id                      KE000063612                          \
parameter           total_precipitation_px0 total_precipitation_px1   
time                                                                  
2024-01-01 00:00:00                0.000000                0.001907   
2024-01-01 01:00:00                0.020981                0.006676   
2024-01-01 02:00:00                0.031948                0.084877   
2024-01-01 03:00:00                0.003815                0.009537   
2024-01-01 04:00:00                0.000000                0.000000   

station_id                                                           \
parameter           total_precipitation_px2 total_precipitation_px3   
time                                                                  
2024-01-01 00:00:00                0.003815                0.000000   
2024-01-01 01:00:00                0.008583                0.010490   
2024-01-01 02:00:00                0.000000                0.002861   
2024-01-01 03:00:00                0.000000                0.000000   
2024-01-01 04:00:00                0.000000                0.000954   

station_id                                                           \
parameter           total_precipitation_px4 total_precipitation_px5   
time                                                                  
2024-01-01 00:00:00                0.000954                0.000954   
2024-01-01 01:00:00                0.015736                0.021458   
2024-01-01 02:00:00                0.001907                0.017166   
2024-01-01 03:00:00                0.001431                0.017643   
2024-01-01 04:00:00                0.000000                0.000000   

station_id                                                           \
parameter           total_precipitation_px6 total_precipitation_px7   
time                                                                  
2024-01-01 00:00:00                0.001907                0.012398   
2024-01-01 01:00:00                0.020027                0.011444   
2024-01-01 02:00:00                0.006676                0.002384   
2024-01-01 03:00:00                0.017166                0.003815   
2024-01-01 04:00:00                0.000477                0.004768   

station_id                                              KE000063619  ...  \
parameter           total_precipitation_px8 total_precipitation_px0  ...   
time                                                                 ...   
2024-01-01 00:00:00                0.000000                     0.0  ...   
2024-01-01 01:00:00                0.004768                     0.0  ...   
2024-01-01 02:00:00                0.004292                     0.0  ...   
2024-01-01 03:00:00                0.012875                     0.0  ...   
2024-01-01 04:00:00                0.000000                     0.0  ...   

station_id                      TZM00063870             ZA000067475  \
parameter           total_precipitation_px8 total_precipitation_px0   
time                                                                  
2024-01-01 00:00:00                0.477791                0.001907   
2024-01-01 01:00:00                0.002384                0.003815   
2024-01-01 02:00:00                0.000477                0.005245   
2024-01-01 03:00:00                0.021935                0.013351   
2024-01-01 04:00:00                0.040054                0.015736   

station_id                                                           \
parameter           total_precipitation_px1 total_precipitation_px2   
time                                                                  
2024-01-01 00:00:00                0.001907                0.004768   
2024-01-01 01:00:00                0.003815                0.007629   
2024-01-01 02:00:00                0.015259                0.017643   
2024-01-01 03:00:00                0.028133                0.027180   
2024-01-01 04:00:00                0.022411      

In [15]:
# preview Imerg
gridded_context['imerg'].head()

station_id                      KE000063612                          \
parameter           total_precipitation_px0 total_precipitation_px1   
time                                                                  
2024-01-01 00:00:00                     0.0                     0.0   
2024-01-01 00:30:00                     0.0                     0.0   
2024-01-01 01:00:00                     0.0                     0.0   
2024-01-01 01:30:00                     0.0                     0.0   
2024-01-01 02:00:00                     0.0                     0.0   

station_id                                                           \
parameter           total_precipitation_px2 total_precipitation_px3   
time                                                                  
2024-01-01 00:00:00                     0.0                     0.0   
2024-01-01 00:30:00                     0.0                     0.0   
2024-01-01 01:00:00                     0.0                     0.0   
2024-01-01 01:30:00                     0.0                     0.0   
2024-01-01 02:00:00                     0.0                     0.0   

station_id                                                           \
parameter           total_precipitation_px4 total_precipitation_px5   
time                                                                  
2024-01-01 00:00:00                     0.0                     0.0   
2024-01-01 00:30:00                     0.0                     0.0   
2024-01-01 01:00:00                     0.0                     0.0   
2024-01-01 01:30:00                     0.0                     0.0   
2024-01-01 02:00:00                     0.0                     0.0   

station_id                                                           \
parameter           total_precipitation_px6 total_precipitation_px7   
time                                                                  
2024-01-01 00:00:00                     0.0                     0.0   
2024-01-01 00:30:00                     0.0                     0.0   
2024-01-01 01:00:00                     0.0                     0.0   
2024-01-01 01:30:00                     0.0                     0.0   
2024-01-01 02:00:00                     0.0                     0.0   

station_id                                              KE000063619  ...  \
parameter           total_precipitation_px8 total_precipitation_px0  ...   
time                                                                 ...   
2024-01-01 00:00:00                     0.0                     0.0  ...   
2024-01-01 00:30:00                     0.0                     0.0  ...   
2024-01-01 01:00:00                     0.0                     0.0  ...   
2024-01-01 01:30:00                     0.0                     0.0  ...   
2024-01-01 02:00:00                     0.0                     0.0  ...   

station_id                      TZM00063870             ZA000067475  \
parameter           total_precipitation_px8 total_precipitation_px0   
time                                                                  
2024-01-01 00:00:00                   0.095                   0.000   
2024-01-01 00:30:00                   0.135                   0.000   
2024-01-01 01:00:00                   0.100                   0.115   
2024-01-01 01:30:00                   0.000                   0.785   
2024-01-01 02:00:00                   0.000                   0.000   

station_id                                                           \
parameter           total_precipitation_px1 total_precipitation_px2   
time                                                                  
2024-01-01 00:00:00                   0.000                   0.000   
2024-01-01 00:30:00                   0.000                   0.000   
2024-01-01 01:00:00                   0.025                   0.005   
2024-01-01 01:30:00                   0.440                   0.105   
2024-01-01 02:00:00                   0.000      

In [16]:
# preview chirps
gridded_context['chirps'].head()

station_id             KE000063612                          \
parameter  total_precipitation_px0 total_precipitation_px1   
time                                                         
2024-01-01                2.622169                6.980968   
2024-01-02                0.000000                0.000000   
2024-01-03                0.000000                0.000000   
2024-01-04                0.000000                0.000000   
2024-01-05                0.000000                0.000000   

station_id                                                  \
parameter  total_precipitation_px2 total_precipitation_px3   
time                                                         
2024-01-01                6.877599                3.595031   
2024-01-02                0.000000                0.000000   
2024-01-03                0.000000                0.000000   
2024-01-04                0.000000                0.000000   
2024-01-05                0.000000                0.000000   

station_id                                                  \
parameter  total_precipitation_px4 total_precipitation_px5   
time                                                         
2024-01-01                4.993625                6.146013   
2024-01-02                0.000000                0.000000   
2024-01-03                0.000000                0.000000   
2024-01-04                0.000000                0.000000   
2024-01-05                0.000000                0.000000   

station_id                                                  \
parameter  total_precipitation_px6 total_precipitation_px7   
time                                                         
2024-01-01                3.374866                4.979064   
2024-01-02                0.000000                0.000000   
2024-01-03                0.000000                0.000000   
2024-01-04                0.000000                0.000000   
2024-01-05                0.000000                0.000000   

station_id                                     KE000063619  ...  \
parameter  total_precipitation_px8 total_precipitation_px0  ...   
time                                                        ...   
2024-01-01                6.140486                     0.0  ...   
2024-01-02                0.000000                     0.0  ...   
2024-01-03                0.000000                     0.0  ...   
2024-01-04                0.000000                     0.0  ...   
2024-01-05                0.000000                     0.0  ...   

station_id             TZM00063870             ZA000067475  \
parameter  total_precipitation_px8 total_precipitation_px0   
time                                                         
2024-01-01                0.000000                6.064589   
2024-01-02                0.000000                9.096883   
2024-01-03                4.510422                6.064589   
2024-01-04               18.041687                3.032294   
2024-01-05               18.041687                6.064589   

station_id                                                  \
parameter  total_precipitation_px1 total_precipitation_px2   
time                                                         
2024-01-01               10.025489                6.974565   
2024-01-02               10.025489                6.974565   
2024-01-03                0.000000                6.974565   
2024-01-04                5.012744                3.487282   
2024-01-05               10.025489                6.974565   

station_id                                                  \
parameter  total_precipitation_px3 total_precipitation_px4   
time                                                         
2024-01-01                6.806942                6.325258   
2024-01-02               10.210412                9.487887   
2024-01-03                3.403471                3.162629   
2024-01-04                3.403471                3.162629   
2024-01-05                6.806942             

In [17]:
# Get the satellite/renalsyis products (lazy loading)
raw_datasets = ds.load_gridded_datasets(groups=["era5"],
                                        years=[2020, 2021, 2022, 2023, 2024])
print(raw_datasets.keys())
print(raw_datasets['era5'])

dict_keys(['era5'])
<xarray.Dataset> Size: 59GB
Dimensions:                       (time: 43848, lat: 69, lon: 57, level: 13)
Coordinates:
  * time                          (time) datetime64[ns] 351kB 2020-01-01 ... ...
  * lat                           (lat) float64 552B 5.0 4.75 ... -11.75 -12.0
  * lon                           (lon) float64 456B 28.0 28.25 ... 41.75 42.0
  * level                         (level) int64 104B 1000 925 850 ... 150 100 50
    number                        int64 8B ...
    step                          timedelta64[ns] 8B ...
Data variables: (12/14)
    10m_u_component_of_wind       (time, lat, lon) float32 690MB dask.array<chunksize=(48, 69, 57), meta=np.ndarray>
    10m_v_component_of_wind       (time, lat, lon) float32 690MB dask.array<chunksize=(48, 69, 57), meta=np.ndarray>
    2m_temperature                (time, lat, lon) float32 690MB dask.array<chunksize=(48, 69, 57), meta=np.ndarray>
    geopotential                  (time, level, lat, lon) float

In [18]:
# load to physical RAM
raw_datasets['era5'][['total_precipitation']].compute()

<xarray.Dataset> Size: 690MB
Dimensions:              (time: 43848, lat: 69, lon: 57)
Coordinates:
  * time                 (time) datetime64[ns] 351kB 2020-01-01 ... 2024-12-3...
  * lat                  (lat) float64 552B 5.0 4.75 4.5 ... -11.5 -11.75 -12.0
  * lon                  (lon) float64 456B 28.0 28.25 28.5 ... 41.5 41.75 42.0
    number               int64 8B 0
    step                 timedelta64[ns] 8B 00:00:00
Data variables:
    total_precipitation  (time, lat, lon) float32 690MB 0.0 0.0 ... 0.1469

In [19]:
# Showcase windows logic
sample_stations = ds.stations.to_list()[:20]
window_vars = ['total_precipitation'] # Adjust based on available TAHMO parameters
start_date = "2020-01-01"
end_date = "2024-12-31"

window_lookback = 24  # 24 hours of historical context
forecast_horizon = 6  # 6 hours of prediction

In [23]:
# Standard Observation Tensors (NumPy)
print("\n--- Standard Numpy Tensors ---")
np_windows = ds.get_windows(
    window_size=window_lookback,
    horizon_size=forecast_horizon,
    stations=sample_stations,
    parameters=window_vars,
    first_date=start_date,
    last_date=end_date,
    as_xarray=False
)

print(f"Features (X) shape:      {np_windows.x.shape}")
print(f"Targets (Y) shape:       {np_windows.y.shape}")
print(f"Feature Mask shape:      {np_windows.mask_x.shape}")
print(f"Target Mask shape:       {np_windows.mask_y.shape}")
print(f"Index X (Reftime) shape: {np_windows.index_x.shape}")


--- Standard Numpy Tensors ---
Features (X) shape:      (43795, 24, 20, 1)
Targets (Y) shape:       (43795, 6, 20, 1)
Feature Mask shape:      (43795, 24, 20, 1)
Target Mask shape:       (43795, 6, 20, 1)
Index X (Reftime) shape: (43795, 24)
